# MySketch — AI Sketch Completion (Diffusers + SDXL + ControlNet)

Этот ноутбук запускает API для дорисовки изображений.
Использует SDXL + ControlNet (scribble) через HuggingFace Diffusers.

После запуска будет выведен публичный URL — его нужно указать в `server/server.py` как `FOOOCUS_URL`.

## ⚙️ Шаг 1: Проверка GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "N/A")

## 📦 Шаг 2: Установка зависимостей

In [ ]:
!pip install -q diffusers transformers accelerate controlnet_aux safetensors \
    fastapi uvicorn nest-asyncio pyngrok pillow httpx

## 🎨 Шаг 3: Загрузка модели SDXL + ControlNet

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, AutoencoderKL
from diffusers.utils import load_image
from PIL import Image
import numpy as np

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
CONTROLNET_ID = "diffusers/controlnet-scribble-sdxl-1.0"
VAE_ID = "madebyollin/sdxl-vae-fp16-fix"

print("Loading ControlNet...")
controlnet = ControlNetModel.from_pretrained(
    CONTROLNET_ID,
    torch_dtype=torch.float16,
).to("cuda")

print("Loading VAE (fp16 fix)...")
vae = AutoencoderKL.from_pretrained(VAE_ID, torch_dtype=torch.float16).to("cuda")

print("Loading SDXL pipeline...")
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    MODEL_ID,
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
).to("cuda")

# Оптимизация памяти для Colab (T4 15GB VRAM)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

print("✅ Model loaded successfully!")

## 🌐 Шаг 4: Запуск API-сервера

In [ ]:
import base64
import io
import logging
import nest_asyncio
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import Response
from pyngrok import ngrok
import uvicorn

# Настрой авторизацию ngrok: https://dashboard.ngrok.com/get-started/your-authtoken
# Раскомментируй и вставь свой токен:
# ngrok.set_auth_token("твой_токен")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("mysketch-colab")

app = FastAPI(title="MySketch Colab API")

# ─── config ───
DEFAULT_PROMPT = (
    "finish this sketch, make it a complete cute illustration, "
    "add creative details, harmonious colors, professional digital art, "
    "smooth shading, beautiful lighting"
)
NEGATIVE_PROMPT = "ugly, deformed, blurry, low quality, bad anatomy, extra limbs"
GUIDANCE_SCALE = 7.5
CONTROLNET_SCALE = 0.8
NUM_STEPS = 30


def preprocess_sketch(image: Image.Image) -> Image.Image:
    """Подготавливает скетч: ч/б, ресайз, инверсия для лучшего распознавания"""
    # Resize to 1024x1024 (SDXL native)
    image = image.resize((1024, 1024), Image.LANCZOS)
    # Convert to grayscale then back to RGB
    if image.mode != "RGB":
        image = image.convert("RGB")
    return image


@app.post("/generate")
async def generate(
    file: UploadFile = File(...),
):
    """Принимает скетч, возвращает дорисованное изображение"""
    
    # 1. Read image
    image_data = await file.read()
    if len(image_data) > 10 * 1024 * 1024:
        return Response(status_code=413, content="File too large")
    
    try:
        sketch = Image.open(io.BytesIO(image_data))
    except Exception:
        return Response(status_code=400, content="Invalid image")
    
    # 2. Preprocess
    control_image = preprocess_sketch(sketch)
    
    # 3. Generate
    logger.info("Generating...")
    result = pipe(
        prompt=DEFAULT_PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        image=control_image,
        controlnet_conditioning_scale=CONTROLNET_SCALE,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_STEPS,
    ).images[0]
    
    # 4. Return
    buf = io.BytesIO()
    result.save(buf, format="PNG")
    logger.info("Done!")
    return Response(content=buf.getvalue(), media_type="image/png")


@app.get("/health")
async def health():
    return {"status": "ok", "model": "SDXL + ControlNet (scribble)"}


# ─── Запуск ───
nest_asyncio.apply()

# Получаем публичный URL через ngrok
public_url = ngrok.connect(8000).public_url
print(f"\n{'='*60}")
print(f"  🔗 Публичный URL: {public_url}")
print(f"{'='*60}")
print(f"  Укажи этот URL в server/server.py:")
print(f"  set FOOOCUS_URL={public_url}")
print(f"{'='*60}\n")

uvicorn.run(app, host="0.0.0.0", port=8000)

## 📝 Как использовать

1. Запусти все ячейки (Runtime → Run all)
2. Дождись загрузки модели (~2-3 минуты на T4)
3. Скопируй публичный URL из вывода
4. В локальном `server/server.py` установи `FOOOCUS_URL` в этот URL
5. Запусти `python server.py` и открывай `http://localhost:8000/`

### ⚠️ Важно
- В Colabe будет работать ~4-6 часов, потом отключится
- Если упала ошибка памяти — перезапусти Runtime (Runtime → Restart runtime)
- Если нужно сбросить — Runtime → Disconnect and delete runtime